In [2]:
import pandas as pd


url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

Pregunta A (Sumarización Categórica): Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?

**R-** La tasa de supervivencia global del barco fue de 38.38% mientras que el 61,62% de los pasajeros fallecieron.

In [3]:
df['Survived'].value_counts(normalize=True) * 100

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64

Pregunta B (Agrupación y Agregación): El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

In [4]:
df.groupby('Sex')['Survived'].mean() * 100

Sex
female    74.203822
male      18.890815
Name: Survived, dtype: float64

**Mujeres:** Sobrevivio el 74.20% \
**Hombres:** Sobrevivio el 18.89% \
Los datos confirman que la evacuacion prioriza a mujeres y niños

Pregunta C: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

In [5]:
fare_q1, fare_q3 = df['Fare'].quantile([0.25, 0.75])
fare_iqr = fare_q3 - fare_q1
limite_superior = fare_q3 + (1.5 * fare_iqr)

outlier_fare = df[df['Fare'] > limite_superior]

print(f"Q1: {fare_q1} \nQ3: {fare_q3} \nIQR: {fare_iqr}")
print(f"Limite superior: {limite_superior}")
print(f"Outlier: {outlier_fare}")
print("Distribucion de clase por outliers")
print(outlier_fare['Pclass'].value_counts())

Q1: 7.9104 
Q3: 31.0 
IQR: 23.0896
Limite superior: 65.6344
Outlier:      PassengerId  Survived  Pclass  \
1              2         1       1   
27            28         0       1   
31            32         1       1   
34            35         0       1   
52            53         1       1   
..           ...       ...     ...   
846          847         0       3   
849          850         1       1   
856          857         1       1   
863          864         0       3   
879          880         1       1   

                                                  Name     Sex   Age  SibSp  \
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
27                      Fortune, Mr. Charles Alexander    male  19.0      3   
31      Spencer, Mrs. William Augustus (Marie Eugenie)  female   NaN      1   
34                             Meyer, Mr. Edgar Joseph    male  28.0      1   
52            Harper, Mrs. Henry Sleeper (Myna Haxtun)  female  49.0      1   


**Limite Superior:** 65.6344 libras
**Cantidad de outliers:** 116 pasajeros pagaron una tarifa superior al limite matematico.
**Clase predominante:** La gran mayoria pertenecia a 1ra Clase (Pclass 1).

Pregunta D: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

In [6]:
media_fare = df['Fare'].mean()
mediana_fare = df['Fare'].median()

print(f"Media fare: {media_fare}")
print(f"Media fare: {mediana_fare}")

Media fare: 32.204207968574636
Media fare: 14.4542


La media sea mas doble que la mediana indica una distribucion asimetrica positiva, se debe a la diferencia tan grande entre los precios de los boletos. \
Para KNN podria afectar tanto la variable Fare por la gran diferencia de precios, donde por el precio puede que solo inicie a hacer agrupaciones por medio de la tarifa que se pago, cuando existen riesgos de que el precio pagado y la clase no tengan nada que ver.

Pregunta E: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?

In [10]:
muestra = df.groupby('Survived', group_keys=False).sample(frac=150 / len(df), random_state=42)

print("Tamanno muestra", len(muestra))
print(muestra['Survived'].value_counts(normalize=True))

Tamanno muestra 150
Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64


Se evita el sesgo de seleccion, al tener clases desbalanceadas el muetreo aleatorio nos ayuda a representar una clase que pueda ser minoria.

Pregunta F: Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

In [7]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Sesgo de atraccion o de supervivencia, si los valores nulos en AGE se relacion con las victimas de 3ra clase, quienes tendian a ser adultos jovenes y trabajadores, calcular el promedio ignorando a los NaN hara que la edad promedio sea distinta d elos no sobrevivientes, lo que distorsiona el peso de AGE dentro del modelo.

Pregunta G: Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.

No se deben eliminar con .drop(), Ya que son casos validos de alto valor, lo que ayuda dentro del modelo para estimar las personas que sobrevivieron, En lugar de eliminarlas se debe aplicar una transformacion logaritmica o utilizar algoritmos de arboles de decision.

Pregunta H: Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

In [17]:
df['Titulo'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Age_Imputed'] = df.groupby('Titulo')['Age'].transform(lambda x: x.fillna(x.median()))

print(df['Age_Imputed'])

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888    21.0
889    26.0
890    32.0
Name: Age_Imputed, Length: 891, dtype: float64


Nos ayudaria a poder generar una distincion entre las edades de las personas, ademas de que ayuda a que se disminuya el error residual y ayuda a generar caracteristicas especificas para predecir la supervivencia infantil frente a la adulta.

Pregunta I: Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

In [20]:
var_survived = df['Survived'].var()
print(f"Varianza de Survived: {var_survived}")

Varianza de Survived: 0.23677221654749742


0.0 Implica que todos los valores de la columna son exactamente iguales, es decir, o todos sobrevivieron o todos murieron. \
No existiria la posibilidad de que sucedan casos de exito o que se encuentren fuera del marco posible hacia el modelo. No existiria variabilidad lo que el modelo solamente podria decir que en un accidente o todos mueren o todos sobreviven.

Pregunta J: Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

In [21]:
conteo_subgrupos = df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()
print(conteo_subgrupos)

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64


Sobreajuste \
Al segmentar en combinaciones con solo 1 o 2 observaciones, el modelo llega a memorizar el desenlace de las personas que tengan esas caracteristicas, por lo que el modelo cuando se enfrente dentro de nuevas caracteristicas no sabra como comportarse.